# Fine-Tuning do Mistral-7B-Instruct-v0.3 no Google Colab (GPU T4)
Este notebook executa o fine-tuning do modelo oficial `mistralai/Mistral-7B-Instruct-v0.3` utilizando QLoRA (4-bit NF4) na GPU gratuita do Google Colab.

**Tempo estimado:** ~25 a 35 minutos
**Ao final:** Baixa automaticamente o arquivo `lora_adapters.zip` contendo a sua camada de ajustes treinada.

In [ ]:
# 1. Verificar GPU T4
!nvidia-smi

In [ ]:
# 2. Instalar dependências
!pip install -q torch transformers peft trl bitsandbytes datasets accelerate

In [ ]:
# 3. Gerar dataset de instrução NR-1
import json, random

SYSTEM_PROMPT = (
    "Você é o vitAI, especialista em governança corporativa de riscos psicossociais no trabalho "
    "e conformidade estrita com a Norma Regulamentadora nº 1 (NR-1) e Portaria MTE nº 1.419/2024."
)

areas = [
    ("Atendimento ao Cliente", 32, 45.0, 60.0, 4, 1.0, 1.5, 0, "SOBRECARGA"),
    ("Logística e Distribuição", 28, 15.0, 5.0, 5, 1.1, 1.4, 0, "LIDERANCA"),
    ("Vendas B2B", 35, 10.0, 5.0, 2, 1.0, 0.8, 3, "ASSEDIO"),
    ("Financeiro", 18, 2.0, 1.0, 0, 1.0, 0.1, 0, "ESTAVEL"),
]


def make_sample(area, hc, va, vh, dem, mdem, qc, den, stype):
    user = f"Analise a área {area} ({hc} pessoas): Atestados: {va:+.1f}%, Horas Extras: {vh:+.1f}%, Demissões: {dem} (Média: {mdem}), Queda Clima: {qc:.1f}, Denúncias: {den}. Emita parecer NR-1 em JSON."
    if stype == "SOBRECARGA":
        resp = {
            "risk_score": 88.0,
            "risk_level": "VERMELHO",
            "probable_factor": "SOBRECARGA_METAS",
            "hipotese": "Sobrecarga severa decorrente de metas.",
            "plano_pgr": "Repactuação de metas e inclusão no PGR.",
        }
    elif stype == "LIDERANCA":
        resp = {
            "risk_score": 68.0,
            "risk_level": "AMARELO",
            "probable_factor": "LIDERANCA_CONFLITO",
            "hipotese": "Pico de saídas associado a nova liderança.",
            "plano_pgr": "Pesquisa direcionada e mediação de RH.",
        }
    elif stype == "ASSEDIO":
        resp = {
            "risk_score": 95.0,
            "risk_level": "VERMELHO",
            "probable_factor": "ASSEDIO",
            "hipotese": "Denúncias graves de conduta.",
            "plano_pgr": "Escalonamento ético sigiloso fora do PGR.",
        }
    else:
        resp = {
            "risk_score": 12.0,
            "risk_level": "VERDE",
            "probable_factor": "ESTAVEL_SEM_FATOR",
            "hipotese": "Operação estável sem desvios.",
            "plano_pgr": "Manter monitoramento de rotina.",
        }
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user},
            {"role": "assistant", "content": json.dumps(resp, ensure_ascii=False)},
        ]
    }


samples = []
for _ in range(800):
    a = random.choice(areas)
    samples.append(make_sample(*a))

with open("train.jsonl", "w", encoding="utf-8") as f:
    for s in samples:
        f.write(json.dumps(s, ensure_ascii=False) + "\n")

print(f"Dataset gerado com {len(samples)} amostras.")

In [ ]:
# 4. Carregar Mistral 7B em 4-bit e treinar com QLoRA
import gc
import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

# Limpa resíduos de memória GPU de execuções anteriores
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model_id = "mistralai/Mistral-7B-Instruct-v0.3"
compute_dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

print("Carregando tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Carregando modelo base diretamente na GPU...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=compute_dtype,
)

# Prepara o modelo 4-bit e padroniza as camadas para evitar bfloat16 no T4
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

dataset = load_dataset("json", data_files={"train": "train.jsonl"})

# No QLoRA 4-bit, fp16=False e bf16=False desativam o GradScaler conflitante do T4
sft_config = SFTConfig(
    output_dir="lora_adapters",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_length=512,
    dataset_text_field="messages",
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    peft_config=lora_config,
    processing_class=tokenizer,
    args=sft_config,
)

trainer.train()
trainer.model.save_pretrained("lora_adapters")
tokenizer.save_pretrained("lora_adapters")
print("Treinamento do Mistral-7B concluído com sucesso!")

In [ ]:
# 5. Compactar e baixar os adaptadores para o seu computador
!zip -r lora_adapters.zip lora_adapters/

from google.colab import files

files.download("lora_adapters.zip")